# 08｜直方圖 (Histogram)：外送訂單配送時間分佈與 SLA 服務承諾檢驗

[![在 Colab 開啟](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roberthsu2003/ai-excel-analytics/blob/main/02_%E7%9C%8B%E6%87%82%E8%B3%87%E6%96%99%E8%88%87%E9%81%B8%E5%B0%8D%E5%9C%96%E8%A1%A8/%E6%B1%BA%E5%AE%9A%E9%81%A9%E5%90%88%E5%9C%96%E8%A1%A8%E7%9A%84%E6%80%9D%E8%80%83%E8%A6%8F%E5%89%87/Colab%E7%AF%84%E4%BE%8B/08_%E7%9B%B4%E6%96%B9%E5%9C%96.ipynb)

## 🎯 學習目標與商業情境

- **商業情境**：外送平台向顧客承諾「45 分鐘內送達 (SLA)」。營運月報顯示「上月全平台平均配送時間 36.5 分鐘」，主管認為表現達標；但客服卻收到大量「等了一小時以上」的一星負評客訴。
- **統計核心問題**：**單一「平均數」到底隱瞞了什麼真相？** 為什麼必須用直方圖切出等寬區間（Bins），看清集中波峰與長尾右偏態？

> 💡 **上課操作指引**：請學生在瀏覽器開啟空白 Colab 筆記本，一邊聽老師講解統計意義，一邊依序複製以下儲存格貼上執行！

In [ ]:
# [儲存格 1] 設定繁體中文字型與專業繪圖風格
%pip install -q mplfonts

import glob
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mplfonts
from matplotlib import font_manager
from mplfonts import use_font

# 載入支援繁體中文的開源字型
for font_file in glob.glob(os.path.join(os.path.dirname(mplfonts.__file__), "fonts", "*.[ot]tf")):
    font_manager.fontManager.addfont(font_file)
sns.set_theme(style="whitegrid")
use_font("Noto Sans CJK SC")
plt.rcParams["axes.unicode_minus"] = False
print("✅ 繁體中文圖表環境準備完成")

In [ ]:
# [儲存格 2] 載入 100 筆訂單配送時間資料，並計算關鍵統計指標
# 模擬數據結構：80 筆順暢常規訂單 (15~42 分) + 20 筆極端長尾延誤訂單 (48~82 分)
np.random.seed(42)
normal_orders = np.random.normal(loc=28, scale=6, size=80).clip(15, 44)
delayed_orders = np.random.uniform(low=46, high=82, size=20)
delivery_times = np.round(np.concatenate([normal_orders, delayed_orders]), 1)

df = pd.DataFrame({"配送時間_分鐘": delivery_times})

# 計算關鍵統計量
mean_val = df["配送時間_分鐘"].mean()
median_val = df["配送時間_分鐘"].median()
std_val = df["配送時間_分鐘"].std()
sla_limit = 45.0
sla_breach_pct = (df["配送時間_分鐘"] > sla_limit).mean() * 100
p90_val = df["配送時間_分鐘"].quantile(0.90)

stats_summary = pd.DataFrame({
    "統計指標": ["樣本數 (N)", "平均數 (Mean)", "中位數 (Median)", "標準差 (Std)", "90% 訂單門檻 (P90)", "超過 45 分鐘違約率 (SLA Breach)"],
    "數值": [f"{len(df)} 筆", f"{mean_val:.1f} 分鐘", f"{median_val:.1f} 分鐘", f"{std_val:.1f} 分鐘", f"{p90_val:.1f} 分鐘", f"{sla_breach_pct:.1f} %"]
})
display(stats_summary)

## 🧭 老師講解重點一：統計指標透露的線索

請觀察上方表格中的兩個關鍵數字：
1. **平均數是 36.5 分鐘**，但 **中位數只有 30.1 分鐘**，兩者相差超過 6 分鐘！
   - **中位數 (30.1 分)**：全體從小排到大正中間那筆，代表 **50% 的顧客在半小時內就拿到餐點**，這是絕大多數人的真實滿意體感。
   - **平均數 (36.5 分)**：為什麼會被拉高？因為平均數很容易被少數「拖了 70、80 分鐘」的極端大單拉偏！這就是統計學著名的 **「右偏態（Right-Skewed / 正偏態）」** 或 **「長尾分佈 (Long-Tail)」**。
2. **45 分鐘 SLA 違約率高達 20%**：這意味著每 5 張訂單就有 1 張讓客人等超過 45 分鐘，這正是大量一星負評的真正來源！

In [ ]:
# [儲存格 3] 繪製直方圖：觀察區間波峰、中位數、平均數與 SLA 警戒線
fig, ax = plt.subplots(figsize=(10, 6))

# 切分成等寬區間 (Bins: 每 10 分鐘一格，從 10 到 90 分鐘)
bins_range = np.arange(10, 91, 10)
n, bins, patches = ax.hist(df["配送時間_分鐘"], bins=bins_range, edgecolor="white", linewidth=1.2)

# 針對超過 SLA 45 分鐘的長尾區間塗上警戒紅色，合格區間塗上專業深藍色
for i in range(len(patches)):
    if bins[i] >= 40:
        patches[i].set_facecolor("#ef4444")  # 紅色警戒
    else:
        patches[i].set_facecolor("#2563eb")  # 藍色正常

# 在每個長條上方標示該區間的訂單筆數
for i in range(len(n)):
    if n[i] > 0:
        ax.text((bins[i] + bins[i+1])/2, n[i] + 0.8, f"{int(n[i])} 筆", 
                ha="center", va="bottom", fontsize=10, fontweight="bold", color="#1e293b")

# 標註中位數、平均數與 SLA 承諾線
ax.axvline(median_val, color="#f97316", linestyle="--", linewidth=2.5, label=f"中位數 (Median): {median_val:.1f} 分鐘")
ax.axvline(mean_val, color="#dc2626", linestyle="-.", linewidth=2.5, label=f"平均數 (Mean): {mean_val:.1f} 分鐘 (被長尾拉偏)")
ax.axvline(sla_limit, color="#15803d", linestyle="-", linewidth=2.5, label=f"SLA 承諾門檻: {sla_limit:.0f} 分鐘 (違約率 20%)")

# 標註說明文字與圖表標題
ax.set_title("直方圖分析：配送時間呈現顯著「右偏態長尾」，20% 訂單嚴重違約", fontsize=14, fontweight="bold", pad=16)
ax.set_xlabel("配送時間區間（分鐘）", fontsize=11, labelpad=8)
ax.set_ylabel("訂單數量（筆）", fontsize=11, labelpad=8)
ax.set_xticks(bins_range)
ax.set_ylim(0, max(n) + 5)
ax.legend(loc="upper right", frameon=True, facecolor="#f8fafc", edgecolor="#cbd5e1", fontsize=10)

plt.tight_layout()
plt.show()

## 📊 老師講解重點二：看圖說商業分析與管理決策

### 1. 圖表視覺呈現的三大事實
- **主體波峰集中在 20～30 分鐘（45 筆）**：代表正常的調度與配送流程非常順暢，絕大多數外送員表現優異。
- **右側拖出一條長長的尾巴（紅色區域，共 20 筆）**：從 50 分鐘一路延伸到 80 分鐘。正是這群「等待超過 50 分鐘的客人」在社群媒體與客服渠道發洩不滿。
- **平均數的致命盲區**：若只看平均數 36.5 分鐘，主管會誤判為全員「勉強及格」；直方圖清晰揭露是 **「少數極端案件拖累了全局」**。

### 2. 商業決策與落地行動建議
- ❌ **錯誤決策**：盲目責怪所有外送員速度慢，或全面調高外送運費以增加人手。
- ✅ **精準營運決策**：
  1. **下鑽長尾根因**：針對這 20 筆超過 45 分鐘的訂單進行交叉分析，檢查是否集中在特定「備餐極慢的餐廳商圈」、暴雨時段、或跨區遠距離派單。
  2. **設立動態預警**：當訂單成立已達 30 分鐘而外送員尚未取餐時，系統自動啟動加成轉單或加派人手。
  3. **服務補償機制**：針對超過 45 分鐘的長尾顧客，系統於送達時自動補發優惠券致歉，在客訴爆發前挽回品牌信任。

> 💡 **直方圖一句訣**：看連續數據的集中輪廓，永遠比對「中位數」與「平均數」，並緊盯長尾處的極端值！